# 03: Model Explainability with SHAP & Counterfactual Simulation
## Campus Placement Predictor

This notebook demonstrates:
1. Global SHAP feature importances and beeswarm summary plots.
2. Local waterfall plots for individual candidate predictions.
3. Exact SHAP additivity verification in margin log-odds.
4. Calibrated conversion from log-odds to percentage-point probability contributions.
5. Actionable what-if counterfactual simulations.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import Image, display

from src.data.loader import DataLoader
from src.explainability.counterfactual import WhatIfSimulator
from src.explainability.shap_global import get_global_feature_importance
from src.explainability.shap_local import explain_student_prediction
from src.models.registry import ModelRegistry
from src.utils.config import load_config

### 1. Load Trained Model & Background Data

In [ ]:
config = load_config('config/config.yaml')
num_cols = config['features']['numerical']
cat_cols = config['features']['categorical']

registry = ModelRegistry()
pipeline, meta = registry.load_model('best_placement_model')

loader = DataLoader()
df = loader.load_synthetic_data(biased=False)
sample_df = df.sample(300, random_state=42)
print('Loaded trained model and sample dataset successfully.')

### 2. Global SHAP Summary & Feature Rankings

In [ ]:
df_imp = get_global_feature_importance(pipeline, sample_df, num_cols, cat_cols)
print('Top 10 Global Features by Mean |SHAP| (Log-Odds Impact):')
display(df_imp.head(10))

display(Image(filename='reports/figures/shap_beeswarm.png'))
display(Image(filename='reports/figures/shap_feature_importance_bar.png'))

### 3. Local Explanation & Mathematical Additivity Verification

In [ ]:
candidate = df.iloc[10].to_dict()
explanation = explain_student_prediction(pipeline, candidate, num_cols, cat_cols)

print(f'Base Probability:   {explanation["base_probability"]:.1%}')
print(f'Model Probability:  {explanation["model_probability"]:.1%}')
print(f'Net Prob Delta:     {explanation["total_prob_delta_points"]:+.1f} points')
print(f'Additivity Verified: {explanation["additivity_check"]["is_additive"]} (Error: {explanation["additivity_check"]["additivity_error"]:.2e})\n')

print('Plain-English Impact Drivers:')
for stmt in explanation['plain_english_summary']:
    print(f'  • {stmt}')

### 4. Actionable Counterfactual What-If Simulation

In [ ]:
simulator = WhatIfSimulator(pipeline)

# What if student solves 200 more DSA problems and completes 1 additional internship?
what_if_res = simulator.simulate(
    candidate,
    {'dsa_problems_solved': candidate['dsa_problems_solved'] + 200, 'internships': candidate['internships'] + 1}
)

print('What-If Simulation Results:')
print(f'  Baseline Probability: {what_if_res["baseline_probability"]:.1%}')
print(f'  Modified Probability: {what_if_res["modified_probability"]:.1%}')
print(f'  Net Gain:             {what_if_res["delta_probability_points"]:+.1f} percentage points')
print(f'  Outcome Shift:        {what_if_res["status_shift"]}')

# Recommended interventions
recs = simulator.recommend_interventions(candidate, target_probability=0.85)
if recs:
    print('\nTop Recommended Interventions to Reach 85% Target:')
    display(pd.DataFrame(recs))